# `ClassBase`

`nematics3d.core.class_base.ClassBase` is the common object foundation used by many Nematics3D classes.

Users normally do **not** construct `ClassBase` directly. Instead, they work with concrete classes that inherit a shared object protocol from it. This tutorial explains that shared protocol: how to identify an unfamiliar object, inspect what it contains, understand attribute prefixes, see what can be modified, and inspect relations to other objects.


## What `ClassBase` is for

Different Nematics3D classes can represent very different things, but they often need the same basic object behavior.

A `ClassBase` descendant can expose a stable name, documented readable attributes, controlled assignment, and relations to other Nematics3D objects. The scientific meaning still belongs to the concrete subclass; `ClassBase` only supplies the common object language.

A useful mental model is:

```text
ClassBase
    └── common Nematics3D object behavior
            ├── identity
            ├── attribute discovery
            ├── controlled assignment
            └── object relations

Concrete subclass
    └── the actual scientific or organizational behavior
```


## Setup

The example below uses `RegistryBase`, a concrete class in `nematics3d.core` that inherits from `ClassBase`. The registry itself is not the subject of this tutorial; it is only a small real object on which the inherited inspection tools can be demonstrated.


In [ ]:
from nematics3d.core import RegistryBase

obj = RegistryBase("example registry", info="Used to demonstrate ClassBase")
obj


## Start with an unfamiliar object

When you receive an unfamiliar Nematics3D object, the first useful question is:

> What kind of object is this?

Use:

```python
obj.show_doc()
```

`show_doc()` displays the class docstring of the **concrete class of the current object**. It does not show the `ClassBase` docstring unless the object itself is a `ClassBase`.


In [ ]:
obj.show_doc()


`show_doc()` is deliberately different from the attribute-inspection methods below. It answers **what this class represents and is for**, before you start inspecting its individual fields.


## What can I read?

Use:

```python
obj.show_readable_attrs()
```

to list the registered user-readable attributes and their descriptions.


In [ ]:
obj.show_readable_attrs()


If one field is unfamiliar, inspect only that field:

```python
obj.show_attr_doc("attribute_name")
```


In [ ]:
obj.show_attr_doc("raw_info")


## Reading the attribute prefixes

Many Nematics3D objects use prefixes to tell you the semantic role of an attribute before you know the details of the class.

| Prefix or form | Meaning |
| --- | --- |
| `raw_...` | Canonical stored input or core data. A shorter alias without `raw_` is normally available for reading. |
| `state_...` | Writable runtime state describing the current state of the object. |
| `default_...` | A managed default-layer value. |
| `calc_...` | A computed result. Normally read-only from the public interface. |
| `entity_...` | A computed or generated object-valued result. Normally read-only. |
| `impl_...` | Internal implementation state. Ordinary users should normally ignore it. |
| no prefix, relation | A semantic link to another object, such as `owner` or `registry`. |
| no prefix, property | A normal Python property; its precise role is documented by the concrete class. |

The prefixes are therefore not cosmetic naming conventions. They let a user quickly distinguish input, mutable state, derived results, generated objects, and internal implementation state.


### `raw_` and the public alias

`raw_` attributes have one additional convenience. If a class exposes `raw_xxx`, `ClassBase` normally also lets you read the same value through `xxx`.

For the example registry:


In [ ]:
obj.raw_info, obj.info


Both names read the same underlying field. The explicit `raw_` form remains useful in documentation because it tells you that this is canonical stored data; the shorter alias is often more convenient in ordinary analysis code.


## What can I modify?

A `ClassBase` object is not an unrestricted Python attribute bag. A concrete class can distinguish writable inputs from computed outputs, protected fields, and fixed core data.

Instead of guessing, use:

```python
obj.show_modifiable_attrs()
```


In [ ]:
obj.show_modifiable_attrs()


This list is instance-aware. A field can exist and be readable without being modifiable on the current object.

When public assignment is allowed, `ClassBase` routes it through the class's validation and protection rules rather than blindly storing an arbitrary value.


In [ ]:
obj.info = "Updated through the public alias"
obj.info


## Object identity

Every `ClassBase` object has a readable name through `name`, backed by `raw_name`.

For example:


In [ ]:
obj.raw_name, obj.name


The public alias can also be used for renaming:

```python
obj.name = "new name"
```

If the object participates in a registry or another naming constraint, the concrete object system can apply those rules during the assignment.


## Relations between objects

`ClassBase` also provides a common way to represent semantic links between objects. Typical examples are `owner` and `registry`.

Use:

```python
obj.show_relations()
```

to display currently bound relations and their targets.


In [ ]:
obj.show_relations()


For objects embedded in a larger object graph, use:

```python
obj.show_relation_tree(depth=2)
```

to follow declared relations recursively.


## A practical inspection workflow

When an unfamiliar Nematics3D object appears in a notebook, the following sequence is a useful default:

```python
obj.show_doc()                  # What is this object?
obj.show_readable_attrs()       # What can I read?
obj.show_attr_doc("...")        # What does one field mean?
obj.show_modifiable_attrs()     # What can I change?
obj.show_relations()            # What other objects is it connected to?
```

You will not always need every step. The point is that the same questions can be asked of many otherwise unrelated Nematics3D classes.


## What `ClassBase` does not do

`ClassBase` does not perform the scientific calculation of its subclasses. It does not know how to fit a plane, smooth a defect line, diagonalize a $Q$ tensor, or render a figure.

It provides the common object protocol underneath those domain-specific classes. Learning `ClassBase` is therefore learning the **common language of Nematics3D objects**, not learning one physical model or numerical algorithm.


# For developers

The user-facing behavior above is implemented through the class-level attribute schema and per-instance relation and assignment state.

Developers defining a new `ClassBase` descendant should follow the established attribute kinds and prefix conventions so the new class participates in the same inspection workflow. Ordinary users do not need to understand that internal machinery in order to use the object.
